# Auto_MPG


## 1. Importing Module

In [ ]:
import pandas as pd
import numpy as np


## 2. Loading data


In [ ]:
dataset = pd.read_csv('../data/auto-mpg.csv')
dataset.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


## 3. Data Preprocessing

In [ ]:
dataset['horsepower'] = pd.to_numeric(dataset['horsepower'], errors='coerce')
dataset['horsepower'].fillna(dataset['horsepower'].mean(), inplace=True)

dataset.drop('car name', axis=1, inplace=True)

origin = dataset.pop('origin')
dataset['USA'] = (origin == 1) * 1.0
dataset['Europe'] = (origin == 2) * 1.0
dataset['Japan'] = (origin == 3) * 1.0

train_data = dataset.sample(frac=0.8, random_state=1)
val_data = dataset.drop(train_data.index)

y_train = train_data.pop('mpg').values.reshape(-1,1)
x_train = train_data.values
y_val = val_data.pop('mpg').values.reshape(-1,1)
x_val = val_data.values

C:\Users\khacb\AppData\Local\Temp\ipykernel_4892\2467214985.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataset['horsepower'].fillna(dataset['horsepower'].mean(), inplace=True)


## 4. Data Standardisation

In [34]:
mean = np.mean(x_train, axis=0)
std = np.std(x_train, axis=0)
x_train = (x_train - mean) / std
x_val = (x_val - mean) / std

## 5. Init W , bias

In [35]:
def init_parameters():
    np.random.seed(1)   
    W1 = np.random.randn(9,64) * 0.01
    b1 = np.zeros((1,64))
    
    W2 = np.random.randn(64,64) * 0.01
    b2 = np.zeros((1,64))

    W3 = np.random.randn(64,1) * 0.01
    b3 = np.zeros((1,1))
    
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2, 'W3': W3, 'b3': b3}
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z> 0).astype(float)

## 6. Train 

In [36]:

p = init_parameters()
learning_rate = 0.003   
num_epochs = 100 

for epoch in range(num_epochs):
    z1 = np.dot(x_train, p['W1']) + p['b1']
    a1 = relu(z1)
    
    z2 = np.dot(a1, p['W2']) + p['b2']
    a2 = relu(z2)
    
    y_hat = np.dot(a2, p['W3']) + p['b3']
    
    loss = np.mean((y_hat - y_train) ** 2)
    
    m= y_train.shape[0] # number of samples
    
    dy_hat = 2 * (y_hat - y_train) / m # Đạo hàm của hàm mất mát theo y_hat
    dW3 = np.dot(a2.T, dy_hat)  # Đạo hàm của hàm mất mát theo W3
    db3 = np.sum(dy_hat, axis=0, keepdims=True) # Đạo hàm của hàm mất mát theo b3
    
    da2 = np.dot(dy_hat, p['W3'].T) # Đạo hàm của hàm mất mát theo a2
    dz2 = da2 * relu_derivative(z2)
    dW2 = np.dot(a1.T, dz2)  # Đạo hàm của hàm mất mát theo W2
    db2 = np.sum(dz2, axis=0, keepdims=True) # Đạo hàm của hàm mất mát theo b2
    
    da1 = np.dot(dz2, p['W2'].T) # Đạo hàm của hàm mất mát theo a1
    dz1 = da1 * relu_derivative(z1)
    dW1 = np.dot(x_train.T, dz1)  # Đạo hàm của hàm mất mát theo W1
    db1 = np.sum(dz1, axis=0, keepdims=True) # Đạo hàm của hàm mất mát theo b1
    
    p['W3'] -= learning_rate * dW3
    p['b3'] -= learning_rate * db3      
    p['W2'] -= learning_rate * dW2
    p['b2'] -= learning_rate * db2
    p['W1'] -= learning_rate * dW1
    p['b1'] -= learning_rate * db1
    
    if epoch % 10 == 0:    # In ra loss mỗi 10 epoch
        print(f'Epoch {epoch}, Loss: {loss}')  

Epoch 0, Loss: 627.1103975678667
Epoch 10, Loss: 562.7071497826222
Epoch 20, Loss: 501.4430088568291
Epoch 30, Loss: 409.34962842736417
Epoch 40, Loss: 37.25466514221501
Epoch 50, Loss: 11.023686205162509
Epoch 60, Loss: 9.871186380219012
Epoch 70, Loss: 9.362796104658889
Epoch 80, Loss: 8.9190201199132
Epoch 90, Loss: 8.526302946406696


## 7. Đánh giá mô hình trên tập Validation

In [37]:
z1_val = np.dot(x_val, p['W1']) + p['b1']
a1_val = relu(z1_val)
z2_val = np.dot(a1_val, p['W2']) + p['b2']
a2_val = relu(z2_val)
y_val_hat = np.dot(a2_val, p['W3']) + p['b3']

val_loss = np.mean((y_val_hat - y_val) ** 2)    ## Tính toán loss trên tập validation
print(f'Validation Loss: {val_loss}')

Validation Loss: 12.107574922292937


## 8. R-Square

In [38]:
ssr = np.sum((y_val - y_val_hat) ** 2)
sst = np.sum((y_val - np.mean(y_val)) ** 2)
r2_score = 1 - (ssr / sst)
print(f'R² Score on Validation Set: {r2_score}')


R² Score on Validation Set: 0.7630956298199061
